[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/79_nearest_dining_solution.ipynb)

# Solution: Nearest Free Dining Spot

Reference solution — merge intervals, then check the block containing `p`.

## 解析

**结论：区间是闭且整数的，先合并重叠/相邻区间；若 `p` 不在任何块内答案为 0，否则落在块 `[l, r]` 内，答案是 `min(p-(l-1), (r+1)-p)`。**

### 为什么要按「间隔 <= 1」合并
区间是**闭整数**区间。`[1,3]` 和 `[4,6]` 在整数意义上是连续的（3 和 4 相邻，中间没有空隙），合起来是一整块 `[1,6]`，位置 4 并不是自由点。所以合并条件不是通常的 `l <= 上一段.r`，而是 **`l <= 上一段.r + 1`**——把「刚好贴住」的相邻区间也并进来。

### 主体逻辑
1. 排序后按 `l <= merged[-1].r + 1` 合并成若干不相交的块。
2. 找 `p` 落在哪个块 `[l, r]`：
   - 不在任何块内 → `p` 本身自由，答案 `0`。
   - 在块内 → 离它最近的自由整数只能是块的两侧 `l-1` 或 `r+1`，取较近者：`min(p-(l-1), (r+1)-p)`。

因为整块都被禁止，块内部不可能有自由点，最近自由点必在块外紧邻处——无需向外逐格搜索。

### 样例说明
- `p=5, [[1,3],[7,8]]`：5 不在任何块 → 0。
- `p=0, [[-2,1],[2,4],[6,9]]`：`[-2,1]` 与 `[2,4]` 间隔 1 合并成 `[-2,4]`，0 在其中，`min(0-(-3), 5-0)=3`。

> 注：原题 OCR 文本里有一组样例（`p=10, [[10,20]]` 标注为 0）与「闭区间」定义自相矛盾（10 明明在 `[10,20]` 内，正确答案应为 1，即走到 9）。这里按题面明确的**闭区间**语义实现，并采用与之一致的两组官方样例。

### 复杂度
排序 `O(n log n)`，合并与查找 `O(n)`。空间 `O(n)`。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List

In [ ]:
# ✅ SOLUTION

class Solution:
    def nearest_free_distance(self, p: int, intervals: List[List[int]]) -> int:
        if not intervals:
            return 0
        iv = sorted([list(x) for x in intervals])
        merged = []
        for l, r in iv:
            if merged and l <= merged[-1][1] + 1:      # overlap OR touch (integer gap 1)
                merged[-1][1] = max(merged[-1][1], r)
            else:
                merged.append([l, r])
        for l, r in merged:
            if l <= p <= r:                             # p sits inside this block
                return min(p - (l - 1), (r + 1) - p)
        return 0                                        # p is already free

In [ ]:
# Demo
sol = Solution()
print(sol.nearest_free_distance(5, [[1, 3], [7, 8]]))            # 0
print(sol.nearest_free_distance(0, [[-2, 1], [2, 4], [6, 9]]))   # 3
print(sol.nearest_free_distance(12, [[10, 20]]))                 # 3

In [ ]:
from torch_judge import check
check('nearest_dining')